# Entrenamiento v3 — detección de daños en palas (OV9281 + Coral)

v3 = v2 + **export a Edge TPU que corre entero en la TPU**. El `.tflite` de v2 corría casi todo en la CPU de la Coral: 0,8 fps a 640×640, porque solo el stem iba a la TPU y quedaban 59 convoluciones en la CPU.

**Qué cambia respecto de v2**

1. **Export en un entorno aislado con versiones fijas** (`/tmp/venv_export`, Python 3.11, en CPU). El problema no estaba en los pesos sino en el entorno: con las versiones de la imagen de Kaggle, onnx2tf metía `TRANSPOSE` (NHWC↔NCHW) alrededor del concat de los 4 C2f del backbone, y el `edgetpu_compiler` corta en el primer operador que no soporta. El mismo `best.pt` exportado con estas versiones (probado en PC) sale sin ningún `TRANSPOSE`.
2. **Cabeza cruda**: el modelo exportado tiene 6 salidas (por nivel: caja DFL de 64 canales + score de clase con sigmoid). El decodificado de la cabeza de Ultralytics (reshape, transpose, softmax) dejaba 16 convoluciones de la cabeza en la CPU. Ahora el decode DFL + NMS se hace en numpy con `yolo_raw.py` (~1 ms) y cada salida tiene su propia escala int8. Probado en PC: **229 operadores, todos soportados por la Edge TPU, 0 `TRANSPOSE`**.
3. **Entrada rectangular `EXPORT_IMGSZ = (416, 640)`**: un tile de 640×400 entra a escala 1:1 (la misma del entrenamiento, 5 mm/px) con ~35 % menos cómputo que 640×640. Con la cámara a 1280×720 usar `(384, 640)` (tiles de 640×384). **No** achicar la imagen: a 10 mm/px el daño promedio queda en ~4 px.
4. **`ENTRENAR = False`**: no re-entrena. Toma el `best.pt` de la corrida anterior, regenera solo val y test (salen idénticos: semillas fijas por foto) y re-exporta + evalúa el int8. Con `True` entrena de cero igual que v2 y después exporta igual.
5. Controles nuevos: orden de clases del DTU original (`damage` = muchas cajas chicas) y cobertura de la TPU con `cobertura_tpu.py` sobre el modelo compilado.

**Cómo correrlo en Kaggle (sin re-entrenar)**

- *Add Input* → dataset `ajifoster3/yolo-annotated-wind-turbines-586x371`.
- *Add Input* → el dataset privado con `WTBs2025_2cls` (el mismo de v2).
- *Add Input* → **Notebook Output** del notebook de entrenamiento v2 (`train-dron-v2`), el que generó el `best.pt`. Se busca solo; si hay varios, fijar `BEST_PT`.
- *Settings* → **Internet on** (instala el entorno de export). La GPU es opcional: solo acelera la evaluación del `best.pt`.
- *Run All*. Al final queda `/kaggle/working/entregables.zip` con el `_edgetpu.tflite`, `yolo_raw.py`, el reporte de cobertura y las métricas fp32 vs int8.

**Criterios**: `cobertura_tpu` tiene que decir *OK: todas las convoluciones corren en la Edge TPU*. Si el mAP50 int8 de damage cae más de ~5 puntos respecto del `best.pt`, re-entrenar con `ENTRENAR = True`.

En la Coral: `bench_fps.py` de la carpeta `Coral/` del repo, con `yolo_raw.py` en la misma carpeta.

Para el experimento de escala (E3) se corre cambiando `GSD_OBJETIVO_MM` a 2.5, 5 y 10 (con `ENTRENAR = True`). Para la comparación color/gris (rama A/B) se cambia `MODO_COLOR`.


In [ ]:
# ============================================================
# 1. Instalación y configuración
# ============================================================
# Versión fija de Ultralytics: la misma del run anterior, para poder repetir resultados.
# (Se quitó el "pip install numpy<2": Ultralytics lo reinstalaba igual y dejaba conflictos.)
import subprocess, sys
subprocess.run([sys.executable, "-m", "pip", "install", "-q", "ultralytics==8.4.171"], check=False)

import os, re, csv, json, glob, math, random, shutil, collections, time, functools
from pathlib import Path
import cv2
import numpy as np
import yaml

# ---------- Datasets ----------
USAR_WTBS = True         # True: train/val = DTU + WTBs2025_2cls. False: solo DTU (comparación)
WTBS_DIR = "/kaggle/input/datasets/juliangiles/dataset-wtg"          # ruta a la carpeta WTBs2025_2cls en /kaggle/input (None = se busca sola)
WTBS_COL_IMG = None      # columnas de manifest.csv con la imagen y la clase original (None = se detectan)
WTBS_COL_CLASE = None
VARIANTES_WTBS = 1       # versiones de cada imagen WTBs en train (ya son muchas imágenes y muy variadas)

# ---------- Datos y simulación de la cámara ----------
SEED = 42
GSD_ORIG_MM = 1.5        # mm/px estimado de las fotos de detalle del dataset (Zenmuse X5S, f=15 mm, a 5-8 m)
GSD_OBJETIVO_MM = 5.0    # mm/px del OV9281: lente 3 mm a 5 m. Experimento E3: probar 2.5 y 10
MODO_COLOR = "gris"      # "gris" = OV9281 (rama B) | "rgb" = referencia color (rama A)
SOLO_DANIO = False       # False: entrena damage + dirt (dirt enseña qué NO es daño). True: solo damage
TILE_W, TILE_H = 640, 400   # mitad del frame 1280x800 del OV9281 (tiling 2x2)
SOLAPE = 0.15
VARIANTES_TRAIN = 3      # versiones distintas (escala, ruido, gris) de cada foto DTU de train
NEG_FRAC = 0.20          # fracción máxima de tiles sin cajas...
NEG_LIMITAR = ("train",) # ...solo en train. En val y test se dejan todos (FP/100 limpios más estable)
MIN_BOX_PX = 4           # cajas con lado mayor < 4 px se descartan (indetectables)
MIN_VISIBLE = 0.40       # una caja cortada por el tile se conserva si queda >= 40 % visible
MIN_VALIDO = 0.50        # un tile NEGATIVO debe tener >= 50 % de imagen real (el resto es relleno)
SPLIT = (0.70, 0.15, 0.15)
ORIGINALES_DIR = None    # opcional: carpeta con las fotos originales de DTU (DJI_XXXX.JPG) en /kaggle/input

# ---------- Entrenamiento ----------
MODELO_BASE = "yolov8n.pt"   # experimento E5: "yolo26n.pt"
EPOCHS, PATIENCE, BATCH, IMGSZ = 200, 50, 32, 640
FRACTION = 1.0               # 0.25 + EPOCHS=60 para iterar rápido
P_MIN = 0.60                 # precisión mínima al elegir el umbral de confianza (en val)
N_CALIB = 300                # imágenes de val (al azar) para calibrar la cuantización int8

# ---------- Modo ----------
ENTRENAR = False             # False: NO entrena. Toma best.pt de una corrida anterior (Add Input -> Notebook Output),
                             # regenera solo val/test, re-exporta y evalúa. True: entrena de cero como en v2.
BEST_PT = None               # ruta a best.pt (None = se busca en /kaggle/input)

# ---------- Export Edge TPU ----------
EXPORT_IMGSZ = (416, 640)    # (alto, ancho) de la entrada en la Coral, múltiplos de 32. 416x640 = tile de 640x400 a
                             # escala 1:1 (la del entrenamiento). Cámara a 1280x720: (384, 640), tiles de 640x384.

WORK = Path("/kaggle/working")
TAG = "dtu+wtbs" if USAR_WTBS else "dtu"
OUT_DS = WORK / f"ds_{TAG}_{MODO_COLOR}_{GSD_OBJETIVO_MM:g}mm"
RUN_NAME = f"{Path(MODELO_BASE).stem}_{TAG}_{MODO_COLOR}_{GSD_OBJETIVO_MM:g}mm"

random.seed(SEED)
np.random.seed(SEED)
print("Corrida:", RUN_NAME)
print(f"Factor de escala: {min(1.0, GSD_ORIG_MM / GSD_OBJETIVO_MM):.3f}  "
      f"(un daño de 29 px pasa a ~{29 * min(1.0, GSD_ORIG_MM / GSD_OBJETIVO_MM):.0f} px)")

In [ ]:
# ============================================================
# 2. Indexar los datasets: DTU (tiles -> fotos originales) y WTBs2025_2cls
# ============================================================
def encontrar_dataset():
    candidatos = [Path("/kaggle/input/datasets/ajifoster3/yolo-annotated-wind-turbines-586x371/NordTank586x371")]
    candidatos += [Path(p).parent for p in glob.glob("/kaggle/input/**/labels", recursive=True)]
    for c in candidatos:
        if "wtbs" in str(c).lower():          # no confundir con el segundo dataset
            continue
        if (c / "images").is_dir() and (c / "labels").is_dir():
            return c
    raise FileNotFoundError("No encontré images/ y labels/ en /kaggle/input. ¿Agregaste el dataset al notebook?")

DATASET = encontrar_dataset()
IMG_DIR, LBL_DIR = DATASET / "images", DATASET / "labels"
print("Dataset DTU:", DATASET)

PATRON = re.compile(r"^(DJI_\d+)_(\d+)_(\d+)$")   # DJI_0201_02_06 -> foto, índice A, índice B
imgs, sin_patron = {}, []
for p in IMG_DIR.iterdir():
    if p.suffix.lower() in {".png", ".jpg", ".jpeg"}:
        if PATRON.match(p.stem):
            imgs[p.stem] = p
        else:
            sin_patron.append(p.name)
print(f"Tiles: {len(imgs)}  (sin formato DJI_XXXX_A_B, ignorados: {len(sin_patron)})")

A = [int(PATRON.match(s).group(2)) for s in imgs]
B = [int(PATRON.match(s).group(3)) for s in imgs]
# La foto 5280x2970 se cortó en 9 columnas x 8 filas: el índice con más valores es la columna.
if (max(A) - min(A)) >= (max(B) - min(B)):
    G_COL, G_FIL, COL0, FIL0 = 2, 3, min(A), min(B)
else:
    G_COL, G_FIL, COL0, FIL0 = 3, 2, min(B), min(A)
print(f"Índice A: {min(A)}..{max(A)}  Índice B: {min(B)}..{max(B)}  ->  columna = grupo {G_COL - 1}")

muestra = cv2.imread(str(next(iter(imgs.values()))))
TH0, TW0 = muestra.shape[:2]            # 371, 586
fotos = collections.defaultdict(list)    # foto -> [(stem, fila, col)]
for s in imgs:
    m = PATRON.match(s)
    fotos[m.group(1)].append((s, int(m.group(G_FIL)) - FIL0, int(m.group(G_COL)) - COL0))
NCOL = max(c for v in fotos.values() for _, _, c in v) + 1
NFIL = max(f for v in fotos.values() for _, f, _ in v) + 1
print(f"Fotos: {len(fotos)}  grilla {NCOL}x{NFIL} tiles de {TW0}x{TH0}  ->  lienzo {NCOL * TW0}x{NFIL * TH0}")

# En las etiquetas originales del DTU 0 = dirt y 1 = damage. Se pasan al orden final (0 = damage, 1 = dirt),
# el mismo de WTBs2025_2cls, para que las dos fuentes no queden con las clases cruzadas.
DTU_A_FINAL = {0: 1, 1: 0}

@functools.lru_cache(maxsize=None)          # el disco de /kaggle/input es lento: cada .txt se lee una vez
def leer_labels(stem):
    f = LBL_DIR / f"{stem}.txt"
    if not f.exists():
        return None
    out = []
    for line in f.read_text().splitlines():
        v = line.split()
        if len(v) >= 5:
            out.append((DTU_A_FINAL[int(v[0])],) + tuple(map(float, v[1:5])))
    return out

con_danio = {f for f, v in fotos.items() if any(leer_labels(s) for s, _, _ in v)}
print(f"Fotos con al menos una etiqueta: {len(con_danio)}  /  solo pala limpia: {len(fotos) - len(con_danio)}")

# Control del orden de clases: ya re-mapeado, damage (0) tiene que ser la clase de muchas cajas chicas y dirt (1)
# la de pocas cajas grandes. Si DTU_A_FINAL estuviera al revés, esto falla.
n_cajas, altos = collections.Counter(), collections.defaultdict(list)
for s in imgs:
    for k, _, _, _, bh in leer_labels(s) or []:
        n_cajas[k] += 1
        altos[k].append(bh * TH0)
print("DTU re-mapeado:", {k: f"{n_cajas[k]} cajas, alto medio {np.mean(altos[k]):.0f} px" for k in sorted(n_cajas)})
assert n_cajas[0] > n_cajas[1] and np.mean(altos[0]) < np.mean(altos[1]), \
    "Tras el re-mapeo la clase 0 no es la de muchas cajas chicas: revisar DTU_A_FINAL"
print("OK: damage = 0 (muchas cajas chicas), dirt = 1 (pocas cajas grandes)")


# ---------- WTBs2025_2cls ----------
EXT_IMG = {".png", ".jpg", ".jpeg", ".bmp"}

def estructura_yolo(raiz):
    """{split: (carpeta de imágenes, carpeta de labels)} si raiz tiene train/val/test en formato YOLO."""
    out = {}
    for sp in ("train", "val", "test"):
        for im, lb in ((raiz / "images" / sp, raiz / "labels" / sp), (raiz / sp / "images", raiz / sp / "labels")):
            if im.is_dir() and lb.is_dir():
                out[sp] = (im, lb)
                break
    return out if len(out) == 3 else None

def encontrar_wtbs():
    if WTBS_DIR:
        cands = [Path(WTBS_DIR)]
        cands += sorted({Path(p).parent for p in glob.glob(f"{WTBS_DIR}/**/labels", recursive=True)} | {Path(p).parent.parent for p in glob.glob(f"{WTBS_DIR}/**/train/labels", recursive=True)})
    else:
        cands = {Path(p).parent for p in glob.glob("/kaggle/input/**/labels", recursive=True)}
        cands |= {Path(p).parent.parent for p in glob.glob("/kaggle/input/**/train/labels", recursive=True)}
        cands = sorted(c for c in cands if "wtbs" in str(c).lower())
    for c in cands:
        e = estructura_yolo(c)
        if e:
            return c, e
    return None, None

def leer_manifest(raiz, stems):
    """Clase original de cada imagen según manifest.csv -> {stem: clase}. {} si no se puede leer."""
    rutas = sorted(set(raiz.rglob("manifest.csv")) | set(raiz.parent.glob("manifest.csv")))
    if not rutas:
        print("  manifest.csv no encontrado: no habrá recall por clase original.")
        return {}
    with open(rutas[0], newline="", encoding="utf-8-sig") as fh:
        filas = list(csv.DictReader(fh))
    if not filas:
        return {}
    cols = list(filas[0].keys())
    col_img = WTBS_COL_IMG or max(cols, key=lambda c: sum(Path(str(f[c])).stem in stems for f in filas))
    if WTBS_COL_CLASE:
        col_cls = WTBS_COL_CLASE
    else:
        claves = ("orig", "clase", "class", "categ", "defect", "tipo")
        cand = sorted((c for c in cols if c != col_img and any(k in c.lower() for k in claves)),
                      key=lambda c: 0 if "orig" in c.lower() else 1)
        col_cls = cand[0] if cand else None
    print(f"  manifest: {rutas[0]}  columnas {cols}  ->  imagen = '{col_img}', clase original = '{col_cls}'")
    if col_cls is None:
        print("  No reconocí la columna de clase: fijar WTBS_COL_CLASE en la configuración.")
        return {}
    return {Path(str(f[col_img])).stem: str(f[col_cls]) for f in filas}

WTBS, WTBS_RAIZ, WTBS_CLASE_ORIG = None, None, {}
if USAR_WTBS:
    WTBS_RAIZ, WTBS = encontrar_wtbs()
    if not WTBS:
        raise FileNotFoundError("USAR_WTBS = True pero no encontré WTBs2025_2cls en /kaggle/input. Agregar el "
                                "dataset, fijar WTBS_DIR, o poner USAR_WTBS = False para entrenar solo con DTU.")
    n = {sp: sum(p.suffix.lower() in EXT_IMG for p in d[0].iterdir()) for sp, d in WTBS.items()}
    print(f"\nDataset WTBs2025: {WTBS_RAIZ}  imágenes {n}")
    stems = {p.stem for d in WTBS.values() for p in d[0].iterdir()}
    WTBS_CLASE_ORIG = leer_manifest(WTBS_RAIZ, stems)
    if WTBS_CLASE_ORIG:
        print("  clases originales:", dict(collections.Counter(WTBS_CLASE_ORIG.values())))

In [ ]:
# ============================================================
# 3. Split 70/15/15 por foto (evita la fuga entre tiles de una misma foto)
# ============================================================
def split_por_foto(lista, seed=SEED, p=SPLIT):
    lista = sorted(lista)
    random.Random(seed).shuffle(lista)
    a, b = round(p[0] * len(lista)), round((p[0] + p[1]) * len(lista))
    return lista[:a], lista[a:b], lista[b:]

# Se divide por separado fotos con y sin daño para que los tres splits tengan de ambas
tr1, va1, te1 = split_por_foto(con_danio)
tr2, va2, te2 = split_por_foto(set(fotos) - con_danio)
SPLITS = {"train": tr1 + tr2, "val": va1 + va2, "test": te1 + te2}
for k, v in SPLITS.items():
    print(f"{k:5s}: {len(v):4d} fotos")
assert not (set(SPLITS["train"]) & set(SPLITS["test"])), "Hay fotos repetidas entre train y test"

In [ ]:
# ============================================================
# 4. Funciones: reconstruir foto, simular OV9281, cortar tiles
# ============================================================
import zlib

NOMBRES = {0: "damage"} if SOLO_DANIO else {0: "damage", 1: "dirt"}
DANIO_ID = 0   # damage es siempre la clase 0 (DTU se re-mapea en leer_labels; WTBs2025_2cls ya viene así)

def unir_cortes(cajas, tol=3):
    """Une cajas de la misma clase que quedaron partidas por el borde entre dos tiles."""
    cajas = [list(c) for c in cajas]
    def toca(p, q):
        for a, b in ((p, q), (q, p)):
            sx = round(a[3] / TW0) * TW0             # costura vertical
            if sx > 0 and abs(a[3] - sx) <= tol and abs(b[1] - sx) <= tol and min(a[4], b[4]) > max(a[2], b[2]):
                return True
            sy = round(a[4] / TH0) * TH0             # costura horizontal
            if sy > 0 and abs(a[4] - sy) <= tol and abs(b[2] - sy) <= tol and min(a[3], b[3]) > max(a[1], b[1]):
                return True
        return False
    cambio = True
    while cambio:
        cambio = False
        for i in range(len(cajas)):
            for j in range(i + 1, len(cajas)):
                a, b = cajas[i], cajas[j]
                if a[0] == b[0] and toca(a, b):
                    cajas[i] = [a[0], min(a[1], b[1]), min(a[2], b[2]), max(a[3], b[3]), max(a[4], b[4])]
                    cajas.pop(j)
                    cambio = True
                    break
            if cambio:
                break
    return cajas

def armar_foto(foto):
    """Lienzo con los tiles en su lugar, máscara de zonas reales y cajas en píxeles de la foto."""
    W, H = NCOL * TW0, NFIL * TH0
    lienzo = np.zeros((H, W, 3), np.uint8)
    valido = np.zeros((H, W), np.uint8)
    cajas = []
    for stem, f, c in fotos[foto]:
        im = cv2.imread(str(imgs[stem]))
        x0, y0 = c * TW0, f * TH0
        h, w = min(im.shape[0], H - y0), min(im.shape[1], W - x0)
        lienzo[y0:y0 + h, x0:x0 + w] = im[:h, :w]
        valido[y0:y0 + h, x0:x0 + w] = 1
        for k, cx, cy, bw, bh in leer_labels(stem) or []:
            cajas.append([k, x0 + (cx - bw / 2) * im.shape[1], y0 + (cy - bh / 2) * im.shape[0],
                          x0 + (cx + bw / 2) * im.shape[1], y0 + (cy + bh / 2) * im.shape[0]])
    if ORIGINALES_DIR:                                # foto original completa, si está disponible
        cand = list(Path(ORIGINALES_DIR).glob(f"{foto}.*"))
        if cand:
            orig = cv2.imread(str(cand[0]))
            if orig is not None and orig.shape[0] >= H and orig.shape[1] >= W:
                lienzo, valido[:] = orig[:H, :W].copy(), 1
    if valido.min() == 0:                             # zonas sin tile (cielo / suelo): relleno suave
        w8, h8 = W // 8, H // 8                       # inpainting a baja resolución + blur, sin bordes duros
        peq = cv2.resize(lienzo, (w8, h8), interpolation=cv2.INTER_AREA)
        hueco = cv2.resize((valido == 0).astype(np.uint8), (w8, h8), interpolation=cv2.INTER_NEAREST)
        rell = cv2.inpaint(peq, hueco, 7, cv2.INPAINT_TELEA)
        rell = cv2.GaussianBlur(cv2.resize(rell, (W, H), interpolation=cv2.INTER_LINEAR), (0, 0), 8)
        lienzo[valido == 0] = rell[valido == 0]
    return lienzo, valido, unir_cortes(cajas)

def a_gris(img, rng, aleatorio):
    """Escala de grises. En train los pesos B,G,R varían: el OV9281 tiene otra respuesta espectral (y ve IR)."""
    w = rng.dirichlet([3, 6, 2]) if aleatorio else np.array([0.114, 0.587, 0.299])
    return (img.astype(np.float32) * w.astype(np.float32)).sum(axis=2)

def degradar(f, rng, fuerte):
    """f: float32 0-255 (HxW o HxWx3). Simula lente + sensor. fuerte=True en train, suave y fijo en val/test."""
    h, w = f.shape[:2]
    f = cv2.GaussianBlur(f, (0, 0), rng.uniform(0.5, 1.2) if fuerte else 0.7)
    if fuerte and rng.random() < 0.3:                 # vibración: blur de movimiento leve
        k = int(rng.integers(3, 6))
        ker = np.zeros((k, k), np.float32)
        ker[k // 2, :] = 1
        M = cv2.getRotationMatrix2D((k / 2 - 0.5, k / 2 - 0.5), float(rng.uniform(0, 180)), 1)
        ker = cv2.warpAffine(ker, M, (k, k))
        f = cv2.filter2D(f, -1, ker / max(ker.sum(), 1e-6))
    if fuerte:                                        # exposición distinta
        f = 255.0 * (np.clip(f, 0, 255) / 255.0) ** rng.uniform(0.7, 1.4)
    yy, xx = np.mgrid[0:h, 0:w].astype(np.float32)    # viñeteo de lente gran angular
    r2 = ((xx - w / 2) / (w / 2)) ** 2 + ((yy - h / 2) / (h / 2)) ** 2
    vig = 1 - (rng.uniform(0.05, 0.25) if fuerte else 0.12) * r2 / 2
    f = f * (vig[..., None] if f.ndim == 3 else vig)
    lectura = rng.uniform(2, 6) if fuerte else 3.0    # ruido de disparo + ruido de lectura
    ruido = rng.standard_normal(f.shape).astype(np.float32) * np.sqrt(np.clip(f, 0, 255) * 0.05 + lectura ** 2)
    return np.clip(f + ruido, 0, 255).astype(np.uint8)

def posiciones(largo, tile, solape=SOLAPE):
    if largo <= tile:
        return [0]
    paso = int(tile * (1 - solape))
    return sorted(set(list(range(0, largo - tile, paso)) + [largo - tile]))

def generar_tiles(lienzo, valido, cajas, split, variante, rng, cont=None):
    """Devuelve [(tile uint8, [(clase, x1, y1, x2, y2)] en px del tile, x0, y0)] al GSD objetivo.

    - Un tile con cajas válidas se conserva siempre, aunque tenga mucho relleno (la pala suele ser una
      franja angosta y los recortes sin pala del DTU se quitaron: MIN_VALIDO solo se aplica a negativos).
    - Un tile sin cajas válidas en el que SÍ se ve un daño (demasiado chico o demasiado cortado para
      etiquetarlo) no se devuelve: no es pala limpia, no sirve como negativo.
    """
    cont = cont if cont is not None else collections.Counter()
    jitter = rng.uniform(0.85, 1.15) if split == "train" else 1.0
    s = min(1.0, GSD_ORIG_MM / GSD_OBJETIVO_MM * jitter)
    img = cv2.resize(lienzo, None, fx=s, fy=s, interpolation=cv2.INTER_AREA) if s < 1 else lienzo
    val = cv2.resize(valido, (img.shape[1], img.shape[0]), interpolation=cv2.INTER_NEAREST)
    # Si la foto escalada es más chica que un tile (p. ej. a 10 mm/px) se completa ANTES de simular el
    # sensor: así el relleno también recibe blur, viñeteo y ruido, y no queda un escalón plano artificial.
    ph, pw = max(0, TILE_H - img.shape[0]), max(0, TILE_W - img.shape[1])
    if ph or pw:
        media = [float(v) for v in img.reshape(-1, img.shape[2]).mean(axis=0)]
        img = cv2.copyMakeBorder(img, 0, ph, 0, pw, cv2.BORDER_CONSTANT, value=media)
        val = cv2.copyMakeBorder(val, 0, ph, 0, pw, cv2.BORDER_CONSTANT, value=0)
    if MODO_COLOR == "gris":
        f = a_gris(img, rng, aleatorio=(split == "train"))
    else:
        f = img.astype(np.float32)
    img = degradar(f, rng, fuerte=(split == "train"))
    H, W = img.shape[:2]
    salida = []
    for y0 in posiciones(H, TILE_H):
        for x0 in posiciones(W, TILE_W):
            labs, danio_sin_etiqueta = [], False
            for k, bx1, by1, bx2, by2 in cajas:
                x1, y1, x2, y2 = bx1 * s - x0, by1 * s - y0, bx2 * s - x0, by2 * s - y0
                cx1, cy1, cx2, cy2 = max(x1, 0), max(y1, 0), min(x2, TILE_W), min(y2, TILE_H)
                if cx2 <= cx1 or cy2 <= cy1:
                    continue                          # la caja no toca este tile
                vw, vh = cx2 - cx1, cy2 - cy1
                # Fracción visible respecto de lo máximo que la caja puede ocupar en un tile: así una mancha
                # más grande que el tile (unida por unir_cortes) no desaparece de todos los tiles.
                area_ref = max(min(x2 - x1, TILE_W) * min(y2 - y1, TILE_H), 1e-6)
                if vw * vh / area_ref < MIN_VISIBLE or max(vw, vh) < MIN_BOX_PX or min(vw, vh) < 1:
                    danio_sin_etiqueta |= (k == DANIO_ID)
                    continue
                if SOLO_DANIO and k != DANIO_ID:
                    continue                          # en modo solo-daño, dirt es fondo
                labs.append((k, cx1, cy1, cx2, cy2))
            if not labs:
                if danio_sin_etiqueta:
                    cont["descartados_danio_sin_etiqueta"] += 1
                    continue
                if val[y0:y0 + TILE_H, x0:x0 + TILE_W].mean() < MIN_VALIDO:
                    cont["descartados_poca_pala"] += 1
                    continue
            salida.append((img[y0:y0 + TILE_H, x0:x0 + TILE_W].copy(), labs, x0, y0))
    return salida

def preparar_wtbs(img, split, rng):
    """WTBs2025: GSD desconocido y variable -> no se re-muestrea ni se corta en tiles; solo gris + sensor."""
    f = a_gris(img, rng, aleatorio=(split == "train")) if MODO_COLOR == "gris" else img.astype(np.float32)
    return degradar(f, rng, fuerte=(split == "train"))

In [ ]:
# ============================================================
# 5. Generar el dataset simulado
# ============================================================
if OUT_DS.exists():
    shutil.rmtree(OUT_DS)
negativos = collections.defaultdict(list)
stats = collections.defaultdict(collections.Counter)
lados_danio = collections.defaultdict(list)
t0 = time.time()

def guardar_labels(ruta, labs, w, h, split):
    with open(ruta, "w") as fh:
        for k, x1, y1, x2, y2 in labs:
            fh.write(f"{k} {(x1 + x2) / 2 / w:.6f} {(y1 + y2) / 2 / h:.6f} {(x2 - x1) / w:.6f} {(y2 - y1) / h:.6f}\n")
            stats[split][NOMBRES[k]] += 1
            if k == DANIO_ID:
                lados_danio[split].append(max(x2 - x1, y2 - y1))

def crear_split(split):
    (OUT_DS / "images" / split).mkdir(parents=True, exist_ok=True)
    (OUT_DS / "labels" / split).mkdir(parents=True, exist_ok=True)

# Sin entrenar, val y test son los mismos que generó la corrida anterior (las semillas son por foto). Si están en
# /kaggle/input (output del notebook de entrenamiento) se copian: generarlos de nuevo tarda más de una hora.
PREVIO = None
if not ENTRENAR:
    cands = [Path(p) for prof in range(1, 5) for p in glob.glob(f"/kaggle/input/{'*/' * prof}{OUT_DS.name}")]
    PREVIO = next((c for c in cands if all((c / "images" / sp).is_dir() for sp in ("val", "test"))), None)
WTBS_CLASE = collections.defaultdict(dict)   # split -> {archivo generado: clase original}

if PREVIO:
    from PIL import Image
    print("Reutilizando val y test de", PREVIO)
    crear_split("train")
    for split in ("val", "test", "test_wtbs"):
        if not (PREVIO / "images" / split).is_dir():
            continue
        for sub in ("images", "labels"):
            shutil.copytree(PREVIO / sub / split, OUT_DS / sub / split, ignore=shutil.ignore_patterns("*.cache"))
        for img in sorted((OUT_DS / "images" / split).glob("*.png")):
            w, h = Image.open(img).size
            lbl = OUT_DS / "labels" / split / f"{img.stem}.txt"
            filas = [l.split() for l in lbl.read_text().splitlines() if l.strip()] if lbl.exists() else []
            for f in filas:
                stats[split][NOMBRES[int(f[0])]] += 1
                if int(f[0]) == DANIO_ID:
                    lados_danio[split].append(max(float(f[3]) * w, float(f[4]) * h))
            stats[split]["tiles"] += 1
            stats[split]["negativos"] += not filas
            stats[split]["imgs_wtbs" if img.name.startswith("WTBS_") else "tiles_dtu"] += 1
            if split == "test_wtbs":
                WTBS_CLASE[split][img.stem] = WTBS_CLASE_ORIG.get(img.stem[len("WTBS_"):].rsplit("_v", 1)[0], "sin dato")
        print(f"  {split}: {stats[split]['tiles']} imágenes  ({time.time() - t0:.0f} s)")
else:
    # ---- 5a. DTU: foto reconstruida -> GSD objetivo -> tiles 640x400 ----
    for split, lista in SPLITS.items():
        crear_split(split)
        if split == "train" and not ENTRENAR:
            continue                                   # sin entrenar no hace falta train; val y test salen idénticos
        n_var = VARIANTES_TRAIN if split == "train" else 1
        for i, foto in enumerate(lista):
            lienzo, valido, cajas = armar_foto(foto)
            for v in range(n_var):
                rng = np.random.default_rng(zlib.crc32(f"{SEED}-{foto}-{split}-{v}".encode()))
                for tile, labs, x0, y0 in generar_tiles(lienzo, valido, cajas, split, v, rng, stats[split]):
                    nombre = f"{foto}_v{v}_{x0}_{y0}"
                    cv2.imwrite(str(OUT_DS / "images" / split / f"{nombre}.png"), tile)
                    guardar_labels(OUT_DS / "labels" / split / f"{nombre}.txt", labs, TILE_W, TILE_H, split)
                    stats[split]["tiles"] += 1
                    stats[split]["tiles_dtu"] += 1
                    if not labs:
                        negativos[split].append(nombre)
            if i % 50 == 0:
                print(f"  DTU {split}: {i}/{len(lista)} fotos  ({time.time() - t0:.0f} s)")

    # ---- 5b. WTBs2025_2cls: imagen completa, solo gris + sensor. Su test va aparte (test_wtbs) ----
    if WTBS:
        for sp, (im_dir, lb_dir) in WTBS.items():
            split = "test_wtbs" if sp == "test" else sp
            crear_split(split)
            if split == "train" and not ENTRENAR:
                continue
            n_var = VARIANTES_WTBS if split == "train" else 1
            rutas = sorted(p for p in im_dir.iterdir() if p.suffix.lower() in EXT_IMG)
            for p in rutas:
                lbl = lb_dir / f"{p.stem}.txt"
                filas = [l.split() for l in lbl.read_text().splitlines()] if lbl.exists() else []
                filas = [f for f in filas if len(f) >= 5]
                if not filas:                              # WTBs no trae pala limpia: sin etiqueta = error de origen
                    stats[split]["wtbs_sin_etiqueta"] += 1
                    continue
                img = cv2.imread(str(p))
                if img is None:
                    stats[split]["wtbs_ilegibles"] += 1
                    continue
                h, w = img.shape[:2]
                labs = []
                for f in filas:
                    k, (cx, cy, bw, bh) = int(f[0]), map(float, f[1:5])
                    if k != 0:                             # en WTBs2025_2cls todo es damage (0)
                        stats[split]["wtbs_clase_inesperada"] += 1
                        continue
                    labs.append((DANIO_ID, max(cx - bw / 2, 0) * w, max(cy - bh / 2, 0) * h,
                                 min(cx + bw / 2, 1) * w, min(cy + bh / 2, 1) * h))
                if not labs:
                    continue
                for v in range(n_var):
                    rng = np.random.default_rng(zlib.crc32(f"{SEED}-wtbs-{p.stem}-{split}-{v}".encode()))
                    nombre = f"WTBS_{p.stem}_v{v}"
                    cv2.imwrite(str(OUT_DS / "images" / split / f"{nombre}.png"), preparar_wtbs(img, split, rng))
                    guardar_labels(OUT_DS / "labels" / split / f"{nombre}.txt", labs, w, h, split)
                    stats[split]["tiles"] += 1
                    stats[split]["imgs_wtbs"] += 1
                    WTBS_CLASE[split][nombre] = WTBS_CLASE_ORIG.get(p.stem, "sin dato")
            print(f"  WTBs {sp} -> {split}: {stats[split]['imgs_wtbs']} imágenes  ({time.time() - t0:.0f} s)")

    # ---- Negativos (solo DTU): en train se limitan a NEG_FRAC del split; en val y test se dejan todos ----
    for split in SPLITS:
        pos = stats[split]["tiles"] - len(negativos[split])
        if split in NEG_LIMITAR:
            max_neg = int(NEG_FRAC / (1 - NEG_FRAC) * pos)
            sobran = negativos[split][:]
            random.Random(SEED).shuffle(sobran)
            for nombre in sobran[max_neg:]:
                (OUT_DS / "images" / split / f"{nombre}.png").unlink()
                (OUT_DS / "labels" / split / f"{nombre}.txt").unlink()
            negativos[split] = sobran[:max_neg]
        stats[split]["negativos"] = len(negativos[split])
        stats[split]["tiles"] = pos + stats[split]["negativos"]

print(f"\nDataset generado en {time.time() - t0:.0f} s -> {OUT_DS}")
for split in [s for s in ("train", "val", "test", "test_wtbs") if s in stats]:
    l = np.array(lados_danio[split]) if lados_danio[split] else np.array([0])
    print(f"{split:9s}: {dict(stats[split])}  |  lado mayor de 'damage' (px): "
          f"p10={np.percentile(l, 10):.0f}  mediana={np.median(l):.0f}  p90={np.percentile(l, 90):.0f}  "
          f"<8px: {np.mean(l < 8) * 100:.0f} %")

YAML = WORK / f"{OUT_DS.name}.yaml"
with open(YAML, "w") as fh:
    yaml.dump({"path": str(OUT_DS), "train": "images/train", "val": "images/val", "test": "images/test",
               "names": NOMBRES, "nc": len(NOMBRES)}, fh, sort_keys=False)
YAML_WTBS = None                     # el test de WTBs se evalúa con otro YAML (no entra al test de decisión)
if stats["test_wtbs"]["tiles"]:
    YAML_WTBS = WORK / f"{OUT_DS.name}_wtbs.yaml"
    with open(YAML_WTBS, "w") as fh:
        yaml.dump({"path": str(OUT_DS), "train": "images/train", "val": "images/val", "test": "images/test_wtbs",
                   "names": NOMBRES, "nc": len(NOMBRES)}, fh, sort_keys=False)
with open(WORK / f"{OUT_DS.name}_splits.json", "w") as fh:
    json.dump(SPLITS, fh, indent=1)
print("YAML:", YAML, "| YAML test WTBs:", YAML_WTBS)

In [ ]:
# ============================================================
# 6. Control visual: ¿la foto se reconstruyó bien? ¿los tiles tienen sentido?
# ============================================================
import matplotlib.pyplot as plt

COLORES = {0: (0, 0, 255), 1: (0, 200, 255)}   # 0 = damage (rojo), 1 = dirt (naranja), en BGR

foto_ej = sorted(con_danio)[0]
lienzo, valido, cajas = armar_foto(foto_ej)
vista = lienzo.copy()
for k, x1, y1, x2, y2 in cajas:
    cv2.rectangle(vista, (int(x1), int(y1)), (int(x2), int(y2)), COLORES.get(k, (255, 0, 0)), 6)
plt.figure(figsize=(14, 8))
plt.imshow(cv2.cvtColor(vista, cv2.COLOR_BGR2RGB))
plt.title(f"{foto_ej} reconstruida: si los tiles no encajan, la grilla fila/columna está invertida")
plt.axis("off"); plt.show()

sp_ej = "train" if ENTRENAR else "val"
ejemplos = sorted((OUT_DS / "labels" / sp_ej).glob("*.txt"))
ejemplos = [e for e in ejemplos if e.stat().st_size > 0]
random.Random(1).shuffle(ejemplos)
fig, ax = plt.subplots(3, 4, figsize=(20, 10))
for a, lbl in zip(ax.flat, ejemplos[:12]):
    im = cv2.imread(str(OUT_DS / "images" / sp_ej / f"{lbl.stem}.png"))
    for line in lbl.read_text().splitlines():
        k, cx, cy, bw, bh = line.split()
        k, cx, cy, bw, bh = int(k), float(cx) * TILE_W, float(cy) * TILE_H, float(bw) * TILE_W, float(bh) * TILE_H
        cv2.rectangle(im, (int(cx - bw / 2), int(cy - bh / 2)), (int(cx + bw / 2), int(cy + bh / 2)),
                      COLORES.get(k, (255, 0, 0)), 2)
    a.imshow(cv2.cvtColor(im, cv2.COLOR_BGR2RGB)); a.set_title(lbl.stem, fontsize=8); a.axis("off")
plt.suptitle(f"Tiles de {sp_ej} simulando el OV9281 a {GSD_OBJETIVO_MM:g} mm/px (rojo = damage, naranja = dirt)")
plt.tight_layout(); plt.savefig(WORK / "preview_tiles.jpg", dpi=80); plt.show()

In [ ]:
# ============================================================
# 7. Entrenamiento (o, con ENTRENAR = False, best.pt de una corrida anterior)
# ============================================================
import torch
from ultralytics import YOLO

N_GPU = torch.cuda.device_count()
DEVICE = 0 if N_GPU >= 1 else "cpu"   # con 2xT4 se puede probar [0, 1] (multi-GPU desde notebook a veces falla)
if ENTRENAR:
    n_train = len(list((OUT_DS / "images" / "train").glob("*.png")))
    CACHE = "ram" if n_train < 6000 else False
    print(f"GPUs: {N_GPU}  device={DEVICE}  tiles de train: {n_train}  cache={CACHE}")

    model = YOLO(MODELO_BASE)
    model.train(
        data=str(YAML), imgsz=IMGSZ, epochs=EPOCHS, patience=PATIENCE, batch=BATCH,
        device=DEVICE, fraction=FRACTION, workers=4, cache=CACHE,
        optimizer="auto", cos_lr=True, close_mosaic=15,
        seed=SEED, deterministic=True,
        # Aumentos: sin color en gris (no hay tono ni saturación); brillo sí
        hsv_h=0.0, hsv_s=0.0 if MODO_COLOR == "gris" else 0.5, hsv_v=0.4,
        # Geometría: la pala aparece en cualquier ángulo; escala acotada (el GSD lo fija la distancia de vuelo)
        degrees=15, translate=0.1, scale=0.3, perspective=0.0005,
        fliplr=0.5, flipud=0.5, mosaic=1.0, mixup=0.0,
        project=str(WORK / "runs"), name=RUN_NAME, exist_ok=True, plots=True,
    )
    RUN_DIR = Path(model.trainer.save_dir)
    BEST = RUN_DIR / "weights" / "best.pt"
    BEST_ORIGEN = str(BEST)
else:
    if BEST_PT:
        cands = [Path(BEST_PT)]
    else:
        cands = [Path(p) for p in glob.glob("/kaggle/input/**/best.pt", recursive=True)]
        if not cands:                                   # salida de v2 solo como zip
            for z in glob.glob("/kaggle/input/**/entregables.zip", recursive=True):
                shutil.unpack_archive(z, WORK / "entregables_previos")
            cands = list((WORK / "entregables_previos").rglob("best.pt"))
        # si hay varios: primero el de esta configuración (RUN_NAME) y el de runs/.../weights
        cands.sort(key=lambda p: (RUN_NAME not in str(p), "weights" not in str(p), str(p)))
    if not cands:
        raise FileNotFoundError("ENTRENAR = False pero no hay best.pt en /kaggle/input. Agregar como input el output "
                                "del notebook de entrenamiento (Add Input -> Notebook Output) o fijar BEST_PT.")
    print("best.pt encontrados:", [str(c) for c in cands])
    BEST_ORIGEN = str(cands[0])
    RUN_DIR = WORK / "runs" / RUN_NAME
    (RUN_DIR / "weights").mkdir(parents=True, exist_ok=True)
    BEST = RUN_DIR / "weights" / "best.pt"            # copia: /kaggle/input es de solo lectura y el export escribe al lado
    shutil.copy(cands[0], BEST)
    ck = torch.load(BEST, map_location="cpu", weights_only=False)
    ta = ck.get("train_args", {})
    print(f"Usando {BEST_ORIGEN}\n  Ultralytics {ck.get('version')}, {ck.get('date')}, epoch {ck.get('epoch')}, "
          f"data={Path(str(ta.get('data'))).name}, imgsz={ta.get('imgsz')}")
    if Path(str(ta.get("data"))).name != YAML.name:
        print(f"  ATENCIÓN: best.pt se entrenó con {Path(str(ta.get('data'))).name} y esta configuración genera "
              f"{YAML.name}. Las métricas no son comparables: revisar la configuración o BEST_PT.")
    assert dict(ck["model"].names) == NOMBRES, f"Clases de best.pt {ck['model'].names} != NOMBRES {NOMBRES}"
    del ck
print("Mejor modelo:", BEST)

In [ ]:
# ============================================================
# 8. Evaluación: métricas estándar en TEST + métricas de inspección (umbral elegido en VAL)
# ============================================================
def resumen_val(m, titulo):
    filas = []
    for i, c in enumerate(m.box.ap_class_index):
        p, r, ap50, ap = m.box.class_result(i)
        filas.append((NOMBRES[int(c)], p, r, ap50, ap))
    print(f"\n{titulo}")
    print(f"{'clase':8s} {'P':>6s} {'R':>6s} {'mAP50':>6s} {'mAP50-95':>8s}")
    for n, p, r, a50, a in filas:
        print(f"{n:8s} {p:6.3f} {r:6.3f} {a50:6.3f} {a:8.3f}")
    return {n: {"P": p, "R": r, "mAP50": a50, "mAP50-95": a} for n, p, r, a50, a in filas}

best = YOLO(str(BEST))
m_test = best.val(data=str(YAML), split="test", imgsz=IMGSZ, batch=16, device=DEVICE,
                  project=str(RUN_DIR), name="test_fp32", exist_ok=True)
res_fp32 = resumen_val(m_test, "TEST (DTU, decide) — modelo float (best.pt)")
res_wtbs_fp32 = None
if YAML_WTBS:
    m_w = best.val(data=str(YAML_WTBS), split="test", imgsz=IMGSZ, batch=16, device=DEVICE,
                   project=str(RUN_DIR), name="test_wtbs_fp32", exist_ok=True)
    res_wtbs_fp32 = resumen_val(m_w, "TEST WTBs2025 (referencia, no decide) — modelo float")

def iou(a, b):
    """a: (N,4), b: (M,4) en xyxy -> (N,M)."""
    if len(a) == 0 or len(b) == 0:
        return np.zeros((len(a), len(b)))
    x1 = np.maximum(a[:, None, 0], b[None, :, 0]); y1 = np.maximum(a[:, None, 1], b[None, :, 1])
    x2 = np.minimum(a[:, None, 2], b[None, :, 2]); y2 = np.minimum(a[:, None, 3], b[None, :, 3])
    inter = np.clip(x2 - x1, 0, None) * np.clip(y2 - y1, 0, None)
    aa = (a[:, 2] - a[:, 0]) * (a[:, 3] - a[:, 1]); ab = (b[:, 2] - b[:, 0]) * (b[:, 3] - b[:, 1])
    return inter / (aa[:, None] + ab[None, :] - inter + 1e-9)

def emparejar(pr, cf, gt, iou_min):
    """Asignación 1 a 1 por confianza: cada predicción acierta como mucho una caja real.
    Devuelve (índices de las cajas reales detectadas, cantidad de falsos positivos)."""
    m = iou(pr, gt)
    usados, fp = set(), 0
    for j in np.argsort(-cf):
        cand = [g for g in np.argsort(-m[j]) if m[j, g] >= iou_min and g not in usados] if len(gt) else []
        if cand:
            usados.add(cand[0])
        else:
            fp += 1
    return usados, fp

UMBRALES = np.round(np.arange(0.05, 0.95, 0.05), 2)

def cubo(lado):
    return "<8 px" if lado < 8 else "8-16 px" if lado < 16 else "16-32 px" if lado < 32 else ">32 px"

def evaluar_inspeccion(modelo, split="test", umbral=None, prefijo=None, clase_de=None,
                       iou_min=0.3, device=DEVICE, batch=16):
    """Recall de damage (total, por tamaño y, si se pasa clase_de, por clase original), precisión y falsos
    positivos cada 100 tiles limpios. Con umbral=None el umbral se ELIGE en este split (máximo recall con
    P >= P_MIN); si se pasa, se usa fijo. Así el umbral sale de val y test se mide sin fuga."""
    rutas = sorted(p for p in (OUT_DS / "images" / split).glob("*.png") if not prefijo or p.name.startswith(prefijo))
    datos = []   # (stem, gt damage (N,4), pred damage (M,4), conf (M,), es_negativo)
    for i in range(0, len(rutas), batch):
        lote = rutas[i:i + batch]
        res = modelo.predict([str(p) for p in lote], imgsz=IMGSZ, conf=0.01, verbose=False, device=device)
        for p, r in zip(lote, res):
            gt, todas = [], 0
            for line in (OUT_DS / "labels" / split / f"{p.stem}.txt").read_text().splitlines():
                k, cx, cy, bw, bh = line.split(); todas += 1
                if int(k) == DANIO_ID:
                    h, w = r.orig_shape
                    cx, cy, bw, bh = float(cx) * w, float(cy) * h, float(bw) * w, float(bh) * h
                    gt.append([cx - bw / 2, cy - bh / 2, cx + bw / 2, cy + bh / 2])
            sel = r.boxes.cls.cpu().numpy().astype(int) == DANIO_ID
            datos.append((p.stem, np.array(gt).reshape(-1, 4), r.boxes.xyxy.cpu().numpy()[sel],
                          r.boxes.conf.cpu().numpy()[sel], todas == 0))
    n_neg = int(sum(d[4] for d in datos))

    def medir(t):
        tp = fp = fn = fp_neg = 0
        detectados = []
        for _, gt, pr, cf, neg in datos:
            s = cf >= t
            usados, f = emparejar(pr[s], cf[s], gt, iou_min)
            tp += len(usados); fp += f; fp_neg += f * neg; fn += len(gt) - len(usados)
            detectados.append(usados)
        P = tp / max(tp + fp, 1); R = tp / max(tp + fn, 1)
        return {"umbral": float(t), "P": P, "R": R, "F1": 2 * P * R / max(P + R, 1e-9),
                "FP_cada_100_limpios": 100 * fp_neg / n_neg if n_neg else None}, detectados

    tabla = [medir(t)[0] for t in UMBRALES]
    if umbral is None:
        validos = [f for f in tabla if f["P"] >= P_MIN]
        umbral = (max(validos, key=lambda f: f["R"]) if validos else max(tabla, key=lambda f: f["F1"]))["umbral"]
        origen = f"elegido en {split}" + (f" ({prefijo}*)" if prefijo else "")
    else:
        origen = "fijo (elegido en val)"
    elegido, detectados = medir(umbral)

    # Recall por tamaño (y por clase original) con el MISMO emparejamiento 1 a 1 del umbral elegido
    cubos = {c: [0, 0] for c in ("<8 px", "8-16 px", "16-32 px", ">32 px")}
    por_clase = collections.defaultdict(lambda: [0, 0])
    for (stem, gt, _, _, _), usados in zip(datos, detectados):
        for g in range(len(gt)):
            c = cubos[cubo(max(gt[g, 2] - gt[g, 0], gt[g, 3] - gt[g, 1]))]
            c[1] += 1; c[0] += int(g in usados)
            if clase_de is not None:
                pc = por_clase[clase_de.get(stem, "sin dato")]
                pc[1] += 1; pc[0] += int(g in usados)

    fp_txt = lambda v: "n/a" if v is None else f"{v:.1f}"
    print(f"\nInspección ({split}{', ' + prefijo + '*' if prefijo else ''}, IoU >= {iou_min}, "
          f"{len(datos)} imágenes, {n_neg} limpias)")
    print(f"{'umbral':>6s} {'P':>6s} {'R':>6s} {'F1':>6s} {'FP/100 limpios':>15s}")
    for f in tabla[::2]:
        print(f"{f['umbral']:6.2f} {f['P']:6.3f} {f['R']:6.3f} {f['F1']:6.3f} {fp_txt(f['FP_cada_100_limpios']):>15s}")
    print(f"\nUmbral {umbral:.2f} [{origen}]  ->  P={elegido['P']:.3f}  R={elegido['R']:.3f}  "
          f"FP/100 limpios={fp_txt(elegido['FP_cada_100_limpios'])}")
    print("Recall de damage por tamaño:", {k: f"{v[0] / max(v[1], 1):.2f} (n={v[1]})" for k, v in cubos.items()})
    if clase_de is not None:
        print("Recall por clase original:",
              {k: f"{v[0] / max(v[1], 1):.2f} (n={v[1]})" for k, v in sorted(por_clase.items())})
    return {"split": split, "prefijo": prefijo, "umbral": float(umbral), "origen_umbral": origen,
            "n_imagenes": len(datos), "n_limpias": n_neg, "tabla": tabla, "elegido": elegido,
            "recall_por_tamano": {k: {"recall": v[0] / max(v[1], 1), "n": v[1]} for k, v in cubos.items()},
            "recall_por_clase_original": ({k: {"recall": v[0] / max(v[1], 1), "n": v[1]}
                                           for k, v in por_clase.items()} if clase_de is not None else None)}

# El umbral se elige en val (solo la parte DTU: mismo dominio que el test de decisión) y se aplica fijo en test.
insp_val_fp32 = evaluar_inspeccion(best, split="val", prefijo="DJI_")
insp_fp32 = evaluar_inspeccion(best, split="test", umbral=insp_val_fp32["umbral"])
insp_wtbs_fp32 = (evaluar_inspeccion(best, split="test_wtbs", umbral=insp_val_fp32["umbral"],
                                     clase_de=WTBS_CLASE["test_wtbs"]) if YAML_WTBS else None)

## 9. Export a la Edge TPU

Las tres celdas siguientes escriben los scripts que usa el export: el decodificador para la Coral (`yolo_raw.py`), el análisis de cobertura (`cobertura_tpu.py`) y el export en el entorno aislado (`export_edgetpu.py`).

In [ ]:
%%writefile /kaggle/working/yolo_raw.py
"""
yolo_raw.py - decodificado en numpy de un YOLOv8 exportado con la cabeza "cruda" (train-model-v3.ipynb).

El modelo tiene 6 salidas int8, dos por nivel (stride 8, 16 y 32), todas NHWC:
  - caja : [1, h, w, 64]  logits DFL (4 lados x 16 bins), sin decodificar
  - clase: [1, h, w, nc]  score de cada clase, ya con sigmoid (0-1)
El orden de las salidas en el .tflite no es fijo: se identifican por forma.

Así la Edge TPU ejecuta TODAS las convoluciones, y en la CPU queda solo esto:
filtrar por score en int8 (sin decuantizar), decodificar las pocas anclas que pasan y NMS.
Compatible con Python 3.7 (Coral / Mendel). Solo necesita numpy.
"""
import numpy as np

REG_MAX = 16


def preparar_salidas(out_details):
    """Agrupa las salidas por nivel -> [(det_caja, det_clase, h, w)], de mayor a menor resolución."""
    niveles = {}
    for d in out_details:
        _, h, w, c = d["shape"]
        niveles.setdefault((h, w), {})["box" if c == 4 * REG_MAX else "cls"] = d
    return [(v["box"], v["cls"], h, w) for (h, w), v in sorted(niveles.items(), reverse=True)]


def decodificar(interp, niveles, H, W, conf=0.25, iou=0.45, max_det=300, nms_fn=None):
    """Lee las salidas de `interp` (después de invoke) -> array (K, 6): x1, y1, x2, y2, score, clase.
    Coordenadas en píxeles de la entrada del modelo (HxW). nms_fn(cajas, scores, clases, iou) opcional."""
    cajas, scores, clases = [], [], []
    for bd, cd, h, w in niveles:
        sc, zp = cd["quantization"]
        q = interp.get_tensor(cd["index"])[0].reshape(h * w, -1)        # (anclas, nc) int8
        qmax = q.max(1)
        idx = np.nonzero(qmax >= int(np.ceil(conf / sc + zp)))[0]     # umbral comparado en int8
        if idx.size == 0:
            continue
        bs, bz = bd["quantization"]
        b = interp.get_tensor(bd["index"])[0].reshape(h * w, 4, REG_MAX)[idx]
        b = (b.astype(np.float32) - bz) * bs
        b = np.exp(b - b.max(2, keepdims=True))                         # softmax DFL por lado
        stride = float(H) / h
        dist = (b / b.sum(2, keepdims=True)).dot(np.arange(REG_MAX, dtype=np.float32)) * stride
        cx = (idx % w + 0.5) * stride
        cy = (idx // w + 0.5) * stride
        cajas.append(np.stack([cx - dist[:, 0], cy - dist[:, 1], cx + dist[:, 2], cy + dist[:, 3]], 1))
        scores.append((qmax[idx].astype(np.float32) - zp) * sc)
        clases.append(q[idx].argmax(1))
    if not cajas:
        return np.zeros((0, 6), np.float32)
    cajas = np.concatenate(cajas).astype(np.float32)        # la cuantización viene en float64: todo a float32
    scores = np.concatenate(scores).astype(np.float32)
    clases = np.concatenate(clases)
    keep = (nms_fn or nms)(cajas, scores, clases, iou)[:max_det]
    return np.concatenate([cajas[keep], scores[keep, None], clases[keep, None].astype(np.float32)], 1)


def nms(cajas, scores, clases, iou):
    """NMS por clase (desplaza las cajas según la clase para resolverlo en una sola pasada)."""
    off = cajas + clases[:, None].astype(np.float32) * 4096.0
    area = (off[:, 2] - off[:, 0]) * (off[:, 3] - off[:, 1])
    order = scores.argsort()[::-1]
    keep = []
    while order.size:
        i = order[0]
        keep.append(i)
        r = order[1:]
        xx1 = np.maximum(off[i, 0], off[r, 0])
        yy1 = np.maximum(off[i, 1], off[r, 1])
        xx2 = np.minimum(off[i, 2], off[r, 2])
        yy2 = np.minimum(off[i, 3], off[r, 3])
        inter = np.clip(xx2 - xx1, 0, None) * np.clip(yy2 - yy1, 0, None)
        order = r[inter / (area[i] + area[r] - inter + 1e-9) < iou]
    return np.array(keep, dtype=np.int64)


def cuantizar_entrada(img_u8, inp):
    """img_u8: HxWx3 uint8 RGB, ya del tamaño de entrada. Con scale=1/255 y zp=-128 el int8 es
    pixel-128, o sea invertir el bit alto: no hace falta pasar a float."""
    sc, zp = inp["quantization"]
    img_u8 = np.ascontiguousarray(img_u8)
    if sc == 0:                                         # entrada sin cuantizar
        return img_u8.astype(inp["dtype"])[None]
    if inp["dtype"] == np.int8 and abs(sc - 1 / 255.0) < 1e-6 and zp == -128:
        return (img_u8 ^ 0x80).view(np.int8)[None]
    if inp["dtype"] == np.uint8 and abs(sc - 1 / 255.0) < 1e-6 and zp == 0:
        return img_u8[None]
    q = np.round(img_u8.astype(np.float32) / 255.0 / sc + zp)
    info = np.iinfo(inp["dtype"])
    return np.clip(q, info.min, info.max).astype(inp["dtype"])[None]


In [ ]:
%%writefile /kaggle/working/cobertura_tpu.py
"""
cobertura_tpu.py — qué parte de un modelo *_edgetpu.tflite corre en la Edge TPU
y qué parte cae en la CPU, SIN necesitar la Coral ni el log del compilador.

Uso:
    pip install flatbuffers
    python cobertura_tpu.py best_full_integer_quant_edgetpu.tflite

Cómo funciona: lee el grafo del .tflite. Todo lo que la Edge TPU ejecuta queda
empaquetado en un único operador 'edgetpu-custom-op'; cualquier otro operador
del grafo corre en la CPU. El compilador mapea en orden y corta en el primer
operador que no soporta: todo lo que viene después va a la CPU.

Objetivo: 0 CONV_2D en CPU. Que el final (decodificado de cajas: RESHAPE,
SOFTMAX, CONCAT, etc.) quede en CPU es aceptable: es poco cómputo.
"""
import io
import json
import struct
import sys
import zipfile
from collections import Counter

import flatbuffers
from flatbuffers import number_types as N
from flatbuffers.table import Table

OPN = {0: "ADD", 1: "AVERAGE_POOL_2D", 2: "CONCATENATION", 3: "CONV_2D",
       4: "DEPTHWISE_CONV_2D", 6: "DEQUANTIZE", 9: "FULLY_CONNECTED", 14: "LOGISTIC",
       17: "MAX_POOL_2D", 18: "MUL", 22: "RESHAPE", 23: "RESIZE_BILINEAR", 25: "SOFTMAX",
       34: "PAD", 39: "TRANSPOSE", 41: "SUB", 45: "STRIDED_SLICE", 49: "SPLIT",
       97: "RESIZE_NEAREST_NEIGHBOR", 114: "QUANTIZE"}
TYPES = {0: "float32", 1: "float16", 2: "int32", 3: "uint8", 4: "int64", 7: "int16", 9: "int8"}


def _root(b):
    return Table(b, flatbuffers.encode.Get(N.UOffsetTFlags.packer_type, b, 0))


def _off(t, i):
    return t.Offset(4 + 2 * i)


def _tab(t, i):
    o = _off(t, i)
    return Table(t.Bytes, t.Indirect(o + t.Pos)) if o else None


def _tabs(t, i):
    o = _off(t, i)
    if not o:
        return []
    n, s = t.VectorLen(o), t.Vector(o)
    return [Table(t.Bytes, t.Indirect(s + 4 * k)) for k in range(n)]


def _nums(t, i, fmt):
    o = _off(t, i)
    if not o:
        return []
    n, s = t.VectorLen(o), t.Vector(o)
    return list(struct.unpack_from("<%d%s" % (n, fmt), t.Bytes, s))


def _str(t, i):
    o = _off(t, i)
    return t.String(o + t.Pos).decode() if o else None


def _scal(t, i, typ):
    o = _off(t, i)
    return t.Get(typ, o + t.Pos) if o else 0


def analyze(path):
    buf = open(path, "rb").read()
    m = _root(buf)
    codes = []
    for oc in _tabs(m, 1):
        b = max(_scal(oc, 0, N.Int8Flags), _scal(oc, 3, N.Int32Flags))
        codes.append(_str(oc, 1) or OPN.get(b, f"builtin_{b}"))
    sg = _tabs(m, 2)[0]
    tensors = _tabs(sg, 0)
    ops = [codes[_scal(op, 0, N.Uint32Flags)] for op in _tabs(sg, 3)]

    def tinfo(k):
        t = tensors[k]
        q = _tab(t, 4)
        return {"shape": _nums(t, 0, "i"), "dtype": TYPES.get(_scal(t, 1, N.Int8Flags), "?"),
                "scale": (_nums(q, 2, "f") or [None])[0] if q else None,
                "zero_point": (_nums(q, 3, "q") or [None])[0] if q else None}

    meta = {}
    if b"PK\x03\x04" in buf:                   # metadatos de Ultralytics
        try:
            z = zipfile.ZipFile(io.BytesIO(buf[buf.index(b"PK\x03\x04"):]))
            meta = json.loads(z.read("metadata.json"))
        except Exception:
            pass
    return ops, [tinfo(k) for k in _nums(sg, 1, "i")], [tinfo(k) for k in _nums(sg, 2, "i")], meta


def main(path):
    ops, ins, outs, meta = analyze(path)
    n_tpu = ops.count("edgetpu-custom-op")
    cpu = [o for o in ops if o != "edgetpu-custom-op"]
    print(f"Archivo: {path}")
    if meta:
        print(f"  Ultralytics {meta.get('version')}  imgsz={meta.get('imgsz')}  "
              f"clases={meta.get('names')}")
        print(f"  datos: {meta.get('description', '').split(' on ')[-1]}")
    for t in ins:
        print(f"  entrada: {t['shape']} {t['dtype']}  scale={t['scale']}  zp={t['zero_point']}")
    for t in outs:
        print(f"  salida : {t['shape']} {t['dtype']}  scale={t['scale']}  zp={t['zero_point']}")

    if n_tpu == 0:
        print("\nNo hay edgetpu-custom-op: este archivo NO está compilado para la Edge TPU.")
        return
    first = ops.index("edgetpu-custom-op")
    print(f"\nBloques en Edge TPU: {n_tpu}   operadores en CPU: {len(cpu)}")
    print(f"  antes del bloque TPU : {dict(Counter(ops[:first]))}")
    print(f"  después del bloque TPU: {dict(Counter(ops[first + 1:]))}")
    if n_tpu > 1:
        print("  (más de un bloque TPU: hay idas y vueltas TPU <-> CPU, también es lento)")
    if cpu:
        corte = next((o for o in ops[first + 1:] if o != "edgetpu-custom-op"), None)
        print(f"  primer operador en CPU después de la TPU: {corte}")

    n_conv = sum(o in ("CONV_2D", "DEPTHWISE_CONV_2D") for o in cpu)
    print()
    if n_conv == 0:
        print("OK: todas las convoluciones corren en la Edge TPU. "
              "Lo que queda en CPU es el postproceso.")
    else:
        print(f"MAL: {n_conv} convoluciones corren en la CPU. "
              f"El modelo va a ser lento en la Coral.")


if __name__ == "__main__":
    if len(sys.argv) < 2:
        raise SystemExit("Uso: python cobertura_tpu.py modelo_edgetpu.tflite")
    for p in sys.argv[1:]:
        main(p)
        print("-" * 70)


In [ ]:
%%writefile /kaggle/working/export_edgetpu.py
"""Exporta best.pt a Edge TPU con la cabeza cruda. Corre en el entorno aislado (/tmp/venv_export).
Uso: python export_edgetpu.py best.pt calib.yaml alto ancho carpeta_salida"""
import importlib.metadata
import json
import shutil
import sys
import types
from pathlib import Path

from ultralytics import YOLO
from ultralytics.nn.modules.head import Detect

pt, data, h, w, out = Path(sys.argv[1]), sys.argv[2], int(sys.argv[3]), int(sys.argv[4]), Path(sys.argv[5])


def cabeza_cruda(self, xs):
    """Por nivel: caja (64 canales DFL, sin decodificar) y score de clase (sigmoid). Sin reshape, concat ni
    transpose, así todo mapea a la Edge TPU. El decode va en numpy (yolo_raw.py)."""
    return tuple(t for i in range(self.nl) for t in (self.cv2[i](xs[i]), self.cv3[i](xs[i]).sigmoid()))


model = YOLO(str(pt))
heads = [m for m in model.model.modules() if isinstance(m, Detect)]
assert len(heads) == 1 and not getattr(heads[0], "end2end", False), "Se esperaba una cabeza Detect de YOLOv8"
heads[0].forward = types.MethodType(cabeza_cruda, heads[0])

carpeta = pt.parent / f"{pt.stem}_saved_model"
shutil.rmtree(carpeta, ignore_errors=True)
try:
    model.export(format="edgetpu", imgsz=(h, w), data=data, device="cpu")
except Exception as e:                       # p. ej. falla del compilador: el int8 sin compilar igual sirve
    print("EXPORT CON ERROR:", repr(e))

out.mkdir(parents=True, exist_ok=True)
tag = f"{pt.stem}_{h}x{w}"
for f in sorted(carpeta.glob("*full_integer_quant*")):
    shutil.copy(f, out / f.name.replace(pt.stem, tag, 1))
    print("->", out / f.name.replace(pt.stem, tag, 1))
vers = {p: importlib.metadata.version(p)
        for p in ("ultralytics", "torch", "tensorflow", "onnx", "onnx2tf", "onnxslim", "onnxruntime")}
(out / f"{tag}_versiones.json").write_text(json.dumps(vers, indent=1))
print("Versiones del export:", vers)


In [ ]:
# ============================================================
# 9. Export a Edge TPU: entorno aislado + cabeza cruda + entrada rectangular
# ============================================================
# Versiones con las que el grafo sale sin TRANSPOSE (probado exportando best.pt en PC el 28/10/2026). Con las de la
# imagen de Kaggle onnx2tf metía TRANSPOSE en los C2f del backbone y solo el stem iba a la TPU (0,8 fps).
VENV = Path("/tmp/venv_export")
VPY = VENV / "bin" / "python"
PINS = ["torch==2.5.1+cpu", "torchvision==0.20.1+cpu", "ultralytics==8.4.171", "tensorflow==2.19.0",
        "tf_keras==2.19.0", "keras==3.15.1", "onnx==1.22.0", "onnx2tf==1.28.8", "onnxslim==0.1.98",
        "onnxruntime==1.31.1", "onnx-graphsurgeon==0.6.1", "sng4onnx==2.0.1", "ai-edge-litert==2.3.0",
        "protobuf==5.29.6", "numpy==2.1.3", "h5py==3.16.0", "ml-dtypes==0.6.0", "flatbuffers==25.12.19"]


def sh(cmd, ruido=("W0000", "I0000", "oneDNN", "cuda_", "absl::", "TF-TRT", "computation placer")):
    """Corre un comando mostrando la salida en vivo, sin el ruido de TensorFlow. Devuelve el código de salida.
    Sin GPU visible: el export tiene que ser en CPU, igual que en la prueba de PC. MPLBACKEND=Agg porque el
    notebook define el backend inline de Jupyter, que no existe en el entorno aislado (rompía el export)."""
    p = subprocess.Popen([str(c) for c in cmd], stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True, errors="replace",
                         env={**os.environ, "CUDA_VISIBLE_DEVICES": "", "TF_CPP_MIN_LOG_LEVEL": "2",
                              "MPLBACKEND": "Agg"})
    for line in p.stdout:
        if not any(r in line for r in ruido):
            print(line, end="")
    return p.wait()


if not VPY.exists():
    t0 = time.time()
    sh([sys.executable, "-m", "pip", "install", "-q", "uv"])
    assert sh([sys.executable, "-m", "uv", "venv", "-q", "-p", "3.11", VENV]) == 0, "No se pudo crear el entorno"
    assert sh([sys.executable, "-m", "uv", "pip", "install", "-q", "-p", VPY, "--index-strategy", "unsafe-best-match",
               "--extra-index-url", "https://download.pytorch.org/whl/cpu", *PINS]) == 0, "Falló la instalación"
    print(f"Entorno de export listo en {time.time() - t0:.0f} s")

# Calibración int8 con una muestra ALEATORIA de val (DTU + WTBs, con y sin daño). Con fraction= Ultralytics
# toma las primeras imágenes en orden alfabético (pocas fotos DTU y nada de WTBs), por eso se arma aparte.
# Importa que haya daños con score alto: si no, el rango int8 del score queda corto y se satura.
for sub in ("images", "labels"):
    shutil.rmtree(OUT_DS / sub / "calib", ignore_errors=True)
    (OUT_DS / sub / "calib").mkdir(parents=True)
cal = sorted((OUT_DS / "images" / "val").glob("*.png"))
random.Random(SEED).shuffle(cal)
for p in cal[:N_CALIB]:
    shutil.copy(p, OUT_DS / "images" / "calib" / p.name)
    shutil.copy(OUT_DS / "labels" / "val" / f"{p.stem}.txt", OUT_DS / "labels" / "calib" / f"{p.stem}.txt")
YAML_CALIB = WORK / f"{OUT_DS.name}_calib.yaml"
with open(YAML_CALIB, "w") as fh:
    yaml.dump({"path": str(OUT_DS), "train": "images/calib", "val": "images/calib",
               "names": NOMBRES, "nc": len(NOMBRES)}, fh, sort_keys=False)
print(f"Calibración int8: {min(N_CALIB, len(cal))} imágenes de val al azar")

H_EXP, W_EXP = EXPORT_IMGSZ
EXP_DIR = WORK / "export"
shutil.rmtree(EXP_DIR, ignore_errors=True)
t0 = time.time()
sh([VPY, WORK / "export_edgetpu.py", BEST, YAML_CALIB, H_EXP, W_EXP, EXP_DIR])
print(f"Export: {time.time() - t0:.0f} s")

TAG_EXP = f"{BEST.stem}_{H_EXP}x{W_EXP}"
TFLITE_INT8 = EXP_DIR / f"{TAG_EXP}_full_integer_quant.tflite"
TFLITE_TPU = EXP_DIR / f"{TAG_EXP}_full_integer_quant_edgetpu.tflite"
print("int8 (CPU):", TFLITE_INT8 if TFLITE_INT8.exists() else "NO SE GENERÓ")
print("Edge TPU  :", TFLITE_TPU if TFLITE_TPU.exists() else "NO SE GENERÓ: revisar el error del compilador arriba")

# Cobertura: cuánto del grafo corre en la TPU (log del compilador + análisis del grafo compilado)
COBERTURA = None
for log in sorted(EXP_DIR.glob("*edgetpu.log")):
    print(f"\n--- {log.name} ---")
    print("\n".join(l for l in log.read_text(errors="ignore").splitlines() if "Number of operations" in l))
if TFLITE_TPU.exists():
    r = subprocess.run([str(VPY), str(WORK / "cobertura_tpu.py"), str(TFLITE_TPU)], capture_output=True, text=True,
                       errors="replace")
    COBERTURA = r.stdout + r.stderr
    (EXP_DIR / f"{TAG_EXP}_cobertura.txt").write_text(COBERTURA)
    print("\n" + COBERTURA)
    if "OK: todas las convoluciones" not in COBERTURA:
        print("ATENCIÓN: quedan convoluciones en la CPU. Log completo del compilador:")
        for log in EXP_DIR.glob("*edgetpu.log"):
            print(log.read_text(errors="ignore")[-6000:])


In [ ]:
# ============================================================
# 10. Evaluación del int8 (la misma aritmética que la Edge TPU) con el decode en numpy de yolo_raw.py
# ============================================================
import types
import torchvision
from ultralytics.utils.metrics import ap_per_class

sys.path.insert(0, str(WORK))
import yolo_raw


def interprete(ruta):
    try:
        from ai_edge_litert.interpreter import Interpreter
    except ImportError:
        try:
            import tensorflow as tf
            Interpreter = tf.lite.Interpreter
        except ImportError:
            subprocess.run([sys.executable, "-m", "pip", "install", "-q", "ai-edge-litert"], check=True)
            from ai_edge_litert.interpreter import Interpreter
    it = Interpreter(model_path=str(ruta), num_threads=os.cpu_count())
    it.allocate_tensors()
    return it


class _Arr:
    """Imita un tensor de torch (.cpu().numpy()) para reutilizar evaluar_inspeccion sin cambios."""
    def __init__(self, a):
        self.a = a

    def cpu(self):
        return self

    def numpy(self):
        return self.a


def nms_torch(cajas, scores, clases, iou):
    return torchvision.ops.batched_nms(torch.from_numpy(cajas), torch.from_numpy(scores),
                                       torch.from_numpy(clases.astype(np.int64)), iou).numpy()


class ModeloCrudo:
    """El .tflite int8 de cabeza cruda con la interfaz de YOLO.predict que usan evaluar_inspeccion y map_crudo.
    Cada imagen se infiere una sola vez con conf 0.001 (como el val de Ultralytics) y queda guardada: un umbral más
    alto es solo un filtro, porque la NMS greedy no cambia las cajas de score alto."""

    def __init__(self, ruta):
        self.it = interprete(ruta)
        self.inp = self.it.get_input_details()[0]
        _, self.H, self.W, _ = self.inp["shape"]
        self.niveles = yolo_raw.preparar_salidas(self.it.get_output_details())
        self.cache = {}

    def _inferir(self, ruta):
        img = cv2.imread(str(ruta))
        h0, w0 = img.shape[:2]
        r = min(self.H / h0, self.W / w0)
        nh, nw = round(h0 * r), round(w0 * r)
        top, left = (self.H - nh) // 2, (self.W - nw) // 2
        lb = np.full((self.H, self.W, 3), 114, np.uint8)          # letterbox centrado, como Ultralytics
        lb[top:top + nh, left:left + nw] = img if (nh, nw) == (h0, w0) else cv2.resize(img, (nw, nh))
        self.it.set_tensor(self.inp["index"], yolo_raw.cuantizar_entrada(lb[..., ::-1], self.inp))
        self.it.invoke()
        det = yolo_raw.decodificar(self.it, self.niveles, self.H, self.W, conf=0.001, iou=0.7, nms_fn=nms_torch)
        det[:, [0, 2]] = ((det[:, [0, 2]] - left) / r).clip(0, w0)
        det[:, [1, 3]] = ((det[:, [1, 3]] - top) / r).clip(0, h0)
        return det, (h0, w0)

    def predict(self, rutas, conf=0.25, **kw):
        out = []
        for p in rutas:
            if str(p) not in self.cache:
                self.cache[str(p)] = self._inferir(p)
            det, shape = self.cache[str(p)]
            d = det[det[:, 4] >= conf]
            out.append(types.SimpleNamespace(orig_shape=shape, boxes=types.SimpleNamespace(
                xyxy=_Arr(d[:, :4]), conf=_Arr(d[:, 4]), cls=_Arr(d[:, 5]))))
        return out


def map_crudo(modelo, split, titulo):
    """P, R, mAP50 y mAP50-95 por clase con el mismo emparejamiento que el val de Ultralytics."""
    iouv = np.linspace(0.5, 0.95, 10)
    tps, confs, pcls, tcls = [], [], [], []
    for p in sorted((OUT_DS / "images" / split).glob("*.png")):
        r = modelo.predict([str(p)], conf=0.001, verbose=False)[0]
        h, w = r.orig_shape
        gt = np.array([[float(v) for v in l.split()[:5]] for l in
                       (OUT_DS / "labels" / split / f"{p.stem}.txt").read_text().splitlines() if l.strip()]).reshape(-1, 5)
        gcls = gt[:, 0].astype(int)
        gxy = np.stack([(gt[:, 1] - gt[:, 3] / 2) * w, (gt[:, 2] - gt[:, 4] / 2) * h,
                        (gt[:, 1] + gt[:, 3] / 2) * w, (gt[:, 2] + gt[:, 4] / 2) * h], 1)
        pb, pc = r.boxes.xyxy.cpu().numpy(), r.boxes.conf.cpu().numpy()
        pk = r.boxes.cls.cpu().numpy().astype(int)
        correcto = np.zeros((len(pb), len(iouv)), bool)
        if len(pb) and len(gt):
            m = iou(gxy, pb) * (gcls[:, None] == pk[None, :])
            for i, t in enumerate(iouv):
                g, j = np.nonzero(m >= t)
                if g.size:
                    pares = np.stack([g, j, m[g, j]], 1)
                    pares = pares[pares[:, 2].argsort()[::-1]]
                    pares = pares[np.unique(pares[:, 1], return_index=True)[1]]
                    pares = pares[np.unique(pares[:, 0], return_index=True)[1]]
                    correcto[pares[:, 1].astype(int), i] = True
        tps.append(correcto); confs.append(pc); pcls.append(pk); tcls.append(gcls)
    res = ap_per_class(np.concatenate(tps), np.concatenate(confs), np.concatenate(pcls), np.concatenate(tcls),
                       names=NOMBRES)
    P, R, AP, clases = res[2], res[3], res[5], res[6]
    filas = [(NOMBRES[int(c)], P[i], R[i], AP[i, 0], AP[i].mean()) for i, c in enumerate(clases)]
    print(f"\n{titulo}")
    print(f"{'clase':8s} {'P':>6s} {'R':>6s} {'mAP50':>6s} {'mAP50-95':>8s}")
    for n, p, r, a50, a in filas:
        print(f"{n:8s} {p:6.3f} {r:6.3f} {a50:6.3f} {a:8.3f}")
    return {n: {"P": p, "R": r, "mAP50": a50, "mAP50-95": a} for n, p, r, a50, a in filas}


res_int8 = res_wtbs_int8 = insp_val_int8 = insp_int8 = insp_wtbs_int8 = None
if TFLITE_INT8.exists():
    q = ModeloCrudo(TFLITE_INT8)
    t0 = time.time()
    res_int8 = map_crudo(q, "test", f"TEST (DTU, decide) — int8 {H_EXP}x{W_EXP} cabeza cruda (lo que corre en la Coral)")
    print(f"({len(q.cache)} imágenes en {time.time() - t0:.0f} s)")
    d = NOMBRES[DANIO_ID]
    caida = (res_fp32[d]["mAP50"] - res_int8[d]["mAP50"]) * 100
    print(f"\nCaída en mAP50 de damage (best.pt -> int8 {H_EXP}x{W_EXP}): {caida:.1f} puntos  "
          f"({'OK' if caida <= 5 else 'más de 5: conviene re-entrenar con ENTRENAR = True'})")
    # El umbral para la Coral se elige con el propio modelo int8, en val (parte DTU), y se aplica fijo en test
    insp_val_int8 = evaluar_inspeccion(q, split="val", prefijo="DJI_")
    insp_int8 = evaluar_inspeccion(q, split="test", umbral=insp_val_int8["umbral"])
    if YAML_WTBS:
        res_wtbs_int8 = map_crudo(q, "test_wtbs", "TEST WTBs2025 (referencia, no decide) — int8")
        insp_wtbs_int8 = evaluar_inspeccion(q, split="test_wtbs", umbral=insp_val_int8["umbral"],
                                            clase_de=WTBS_CLASE["test_wtbs"])
else:
    print("No hay modelo int8: revisar el error del export.")


In [ ]:
# ============================================================
# 11. Entregables para descargar
# ============================================================
ENT = WORK / "entregables" / RUN_NAME
ENT.mkdir(parents=True, exist_ok=True)
copiar = [BEST, YAML, WORK / f"{OUT_DS.name}_splits.json", WORK / "preview_tiles.jpg",
          RUN_DIR / "results.csv", RUN_DIR / "results.png"]
copiar += list(RUN_DIR.glob("*PR_curve.png")) + list(RUN_DIR.glob("confusion_matrix*.png"))
copiar += [TFLITE_TPU, TFLITE_INT8, WORK / "yolo_raw.py", WORK / "cobertura_tpu.py"]
copiar += list(EXP_DIR.glob("*.log")) + list(EXP_DIR.glob("*.txt")) + list(EXP_DIR.glob("*.json"))
for p in copiar:
    if Path(p).exists():
        shutil.copy(p, ENT / Path(p).name)

config = {k: globals()[k] for k in ["SEED", "USAR_WTBS", "VARIANTES_WTBS", "GSD_ORIG_MM", "GSD_OBJETIVO_MM",
                                    "MODO_COLOR", "SOLO_DANIO", "NOMBRES", "TILE_W", "TILE_H", "SOLAPE",
                                    "VARIANTES_TRAIN", "NEG_FRAC", "NEG_LIMITAR", "MIN_BOX_PX", "MIN_VISIBLE",
                                    "MIN_VALIDO", "MODELO_BASE", "EPOCHS", "PATIENCE", "BATCH", "IMGSZ",
                                    "FRACTION", "P_MIN", "N_CALIB", "ENTRENAR", "EXPORT_IMGSZ"]}
umbral_coral = (insp_val_int8 or insp_val_fp32)["umbral"]
with open(ENT / "metricas.json", "w") as fh:
    json.dump({"config": config,
               "umbral_sugerido": {"int8_coral": insp_val_int8["umbral"] if insp_val_int8 else None,
                                   "fp32": insp_val_fp32["umbral"], "elegido_en": "val (solo DTU)"},
               "export": {"best_pt": BEST_ORIGEN, "imgsz": EXPORT_IMGSZ, "cabeza": "cruda (6 salidas, yolo_raw.py)",
                          "versiones": PINS, "cobertura_tpu": COBERTURA},
               "test_fp32": res_fp32, "test_int8": res_int8, "test_wtbs_fp32": res_wtbs_fp32,
               "test_wtbs_int8": res_wtbs_int8,
               "inspeccion_val_fp32": insp_val_fp32, "inspeccion_fp32": insp_fp32,
               "inspeccion_val_int8": insp_val_int8, "inspeccion_int8": insp_int8,
               "inspeccion_wtbs_fp32": insp_wtbs_fp32, "inspeccion_wtbs_int8": insp_wtbs_int8,
               "dataset": {k: dict(v) for k, v in stats.items()}}, fh, indent=1, default=float)

archivo = shutil.make_archive(str(WORK / "entregables"), "zip", WORK / "entregables")
print("Listo:", archivo)
print("Contenido:", sorted(p.name for p in ENT.iterdir()))
print(f"\nUmbral sugerido para la Coral: {umbral_coral:.2f} (usar como CONF en 'Prueba modelo.ipynb').")
print(f"Para la Coral: {TFLITE_TPU.name} + yolo_raw.py + bench_fps.py (carpeta Coral/ del repo), p. ej.\n"
      f"  mdt push {TFLITE_TPU.name} /home/mendel/prueba_fps/  y en la placa:  python3 bench_fps.py {TFLITE_TPU.name}")
print("Para probar en la PC: copiar best.pt a 'Proyecto final/Modelo/Modelo nuevo/best.pt' y correr 'Prueba modelo.ipynb'.")